# AlphaZero Best Model Showcase

Este notebook serve apenas para exibir o melhor `AlphaZero` treinado. Ele carrega a melhor run disponível, resume a corrida, mostra curvas, corre uma avaliação final e no fim permite jogar humano contra o modelo.


## Passo 1: Setup

A célula seguinte encontra a raiz do projeto, carrega a configuração, prepara o device e importa as utilidades necessárias para análise e demonstração.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Could not find project root containing config.yaml")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / "outputs"
ALPHAZERO_OUTPUTS = ROOT / "notebooks" / "alphazero" / "outputs"
ALPHAZERO_OUTPUTS.mkdir(parents=True, exist_ok=True)

from connect4_rl.agents.baselines import HeuristicAgent, RandomAgent
from connect4_rl.config import load_config
from connect4_rl.envs.connect_four import apply_action, initial_state, is_terminal, legal_actions, render_ascii
from connect4_rl.experiments import build_agent_from_checkpoint, find_best_run

CONFIG = load_config(ROOT / "config.yaml")
NOTEBOOK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if NOTEBOOK_DEVICE == "cuda":
    torch.set_default_device(NOTEBOOK_DEVICE)

print({
    "torch_version": torch.__version__,
    "device": NOTEBOOK_DEVICE,
    "outputs": str(ALPHAZERO_OUTPUTS),
})


## Passo 2: Escolher a run

Podes apontar explicitamente para uma run, ou deixar o notebook escolher automaticamente a melhor run `AlphaZero` disponível em `outputs/`.


In [ ]:
run_name: str | None = None

agent = None
metrics_data: dict[str, object] = {
    "best_checkpoint_path": None,
    "best_score": None,
    "evaluation": [],
    "tactical_accuracy": [],
    "episode_rewards": [],
    "policy_losses": [],
    "value_losses": [],
}
metrics_path = None

if run_name is None:
    best_run = find_best_run(ALPHAZERO_OUTPUTS, "alphazero")
    if best_run is not None:
        metrics_path = best_run.metrics_path
else:
    candidate = ALPHAZERO_OUTPUTS / run_name / "metrics_final.json"
    if candidate.exists():
        metrics_path = candidate

if metrics_path is not None:
    metrics_data = json.loads(metrics_path.read_text(encoding="utf-8"))
    focus_checkpoint_path = metrics_data.get("best_checkpoint_path")
    if focus_checkpoint_path:
        agent = build_agent_from_checkpoint(
            "alphazero",
            focus_checkpoint_path,
            metrics_data.get("config", {}),
            device=NOTEBOOK_DEVICE,
        )

print({
    "metrics_path": str(metrics_path) if metrics_path is not None else None,
    "focus_checkpoint_path": metrics_data.get("best_checkpoint_path"),
    "best_score": metrics_data.get("best_score"),
    "num_evaluations": len(metrics_data.get("evaluation", [])),
})

if agent is None:
    print("No AlphaZero run found under notebooks/alphazero/outputs/. Showcase cells will stay passive until a run exists.")


## Passo 3: Resumo da run

Mostramos o resumo principal da corrida, o melhor ponto de avaliação e a última avaliação disponível.


In [ ]:
evaluation = metrics_data.get("evaluation", [])
tactical = metrics_data.get("tactical_accuracy", [])
last_eval = evaluation[-1] if evaluation else {}
best_eval = max(evaluation, key=lambda item: float(item.get("vs_heuristic_win_rate", float("-inf")))) if evaluation else {}

print("Run summary")
print({
    "best_checkpoint_path": metrics_data.get("best_checkpoint_path"),
    "best_score": metrics_data.get("best_score"),
    "num_evaluations": len(evaluation),
    "num_tactical_points": len(tactical),
})

print("\nLast evaluation point")
print(last_eval)

print("\nBest heuristic evaluation point")
print(best_eval)


## Passo 4: Curvas

As curvas ajudam a perceber a evolução do modelo ao longo do treino.


In [ ]:
if evaluation:
    eval_episodes = [int(item["episode"]) for item in evaluation]
    vs_random = [float(item.get("vs_random_win_rate", 0.0)) for item in evaluation]
    vs_heuristic = [float(item.get("vs_heuristic_win_rate", 0.0)) for item in evaluation]
    vs_previous = [float(item.get("vs_previous_win_rate", 0.0)) for item in evaluation]
    tactical_x = [int(item["episode"]) for item in tactical]
    tactical_y = [float(item.get("accuracy", 0.0)) for item in tactical]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(eval_episodes, vs_random, marker="o", label="vs random")
    axes[0, 0].plot(eval_episodes, vs_heuristic, marker="o", label="vs heuristic")
    axes[0, 0].plot(eval_episodes, vs_previous, marker="o", label="vs previous")
    axes[0, 0].set_ylim(0.0, 1.0)
    axes[0, 0].set_title("Win rates at evaluation checkpoints")
    axes[0, 0].grid(alpha=0.2)
    axes[0, 0].legend()

    axes[0, 1].plot(metrics_data.get("episode_rewards", []), label="episode reward")
    axes[0, 1].set_title("Episode rewards")
    axes[0, 1].grid(alpha=0.2)

    axes[1, 0].plot(metrics_data.get("policy_losses", []), label="policy loss")
    axes[1, 0].plot(metrics_data.get("value_losses", []), label="value loss")
    axes[1, 0].set_title("Training losses")
    axes[1, 0].grid(alpha=0.2)
    axes[1, 0].legend()

    if tactical:
        axes[1, 1].plot(tactical_x, tactical_y, marker="o", label="tactical accuracy")
    axes[1, 1].set_ylim(0.0, 1.0)
    axes[1, 1].set_title("Tactical accuracy")
    axes[1, 1].grid(alpha=0.2)
    if tactical:
        axes[1, 1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("No evaluation history available.")


## Passo 5: Avaliação final

Corremos uma avaliação mais estável do melhor agente contra `random` e `heuristic`.


In [ ]:
def evaluate_agent(agent, opponent_factory, games: int = 40) -> float:
    wins = 0
    for game_idx in range(games):
        controlled_player = 1 if game_idx % 2 == 0 else 2
        opponent = opponent_factory(game_idx)
        state = initial_state()
        while not is_terminal(state):
            if state.current_player == controlled_player:
                action = agent.select_action(state, legal_actions(state))
            else:
                action = opponent.select_action(state, legal_actions(state))
            state = apply_action(state, action)
        if state.winner == controlled_player:
            wins += 1
    return wins / games

if agent is not None:
    summary_eval = {
        "vs_random": evaluate_agent(agent, lambda idx: RandomAgent(seed=10_000 + idx), games=40),
        "vs_heuristic": evaluate_agent(agent, lambda idx: HeuristicAgent(seed=20_000 + idx), games=40),
    }
    summary_eval
else:
    print("No AlphaZero checkpoint loaded.")


## Passo 6: Partidas exemplo

Imprimimos algumas partidas para inspeção qualitativa do comportamento do agente.


In [ ]:
def play_and_render(agent, opponent, controlled_player: int = 1) -> str:
    state = initial_state()
    transcript = ["Initial board", render_ascii(state), ""]
    move_idx = 0
    while not is_terminal(state):
        move_idx += 1
        if state.current_player == controlled_player:
            action = agent.select_action(state, legal_actions(state))
            actor = agent.name
        else:
            action = opponent.select_action(state, legal_actions(state))
            actor = opponent.name
        state = apply_action(state, action)
        transcript.append(f"Move {move_idx}: {actor} played column {action}")
        transcript.append(render_ascii(state))
        transcript.append("")
    transcript.append(f"Winner: {state.winner}")
    return "\n".join(transcript)

if agent is not None:
    print(play_and_render(agent, RandomAgent(seed=1), controlled_player=1))
else:
    print("No AlphaZero checkpoint loaded.")


In [ ]:
if agent is not None:
    print(play_and_render(agent, HeuristicAgent(seed=2), controlled_player=1))
else:
    print("No AlphaZero checkpoint loaded.")


## Passo 7: Humano vs modelo

No fim, podes jogar manualmente contra o melhor `AlphaZero`.


In [ ]:
def play_human_vs_model(agent, human_player: int = 1) -> None:
    if human_player not in (1, 2):
        raise ValueError("human_player must be 1 or 2")

    state = initial_state()
    print("Starting game")
    print(render_ascii(state))

    while not is_terminal(state):
        legal = legal_actions(state)
        print(f"Current player: {state.current_player} | Legal actions: {legal}")

        if state.current_player == human_player:
            while True:
                raw = input("Choose column: ").strip()
                if not raw.isdigit():
                    print("Please enter an integer column index.")
                    continue
                action = int(raw)
                if action not in legal:
                    print(f"Illegal move. Choose one of {legal}.")
                    continue
                actor = "human"
                break
        else:
            action = agent.select_action(state, legal)
            actor = agent.name

        state = apply_action(state, action)
        print()
        print(f"{actor} played column {action}")
        print(render_ascii(state))
        print()

    if state.winner == 0:
        print("Draw.")
    elif state.winner == human_player:
        print("Human wins.")
    else:
        print("Model wins.")

# Example:
# play_human_vs_model(agent, human_player=1)


In [ ]:
# Example:
# play_human_vs_model(agent, human_player=1)
